# HW01-C — Airflow Scheduled Pipeline

A pipeline that only runs when you remember to click it is a chore.

Here you turn the SQL work into an Airflow DAG. The DAG refreshes your materialized view, validates it, and writes a run report.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- Airflow DAGs: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html
- Airflow Variables: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/variables.html
- Airflow best practices: https://airflow.apache.org/docs/apache-airflow/stable/best-practices.html

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [8]:
from pathlib import Path
import os, re, textwrap

PROJECT = Path.cwd()
for path in ['dags', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(exist_ok=True)

student_id = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', student_id.lower())
DAG_ID = f'qbc12_hw01_{safe_student}_airbnb_pipeline'
STUDENT_SCHEMA = f'student_{safe_student}'
DAG_ID, STUDENT_SCHEMA

('qbc12_hw01_alirezaabavi_airbnb_pipeline', 'student_alirezaabavi')

## 1. DAG design

Build this chain:

```text
read_config → refresh_summary → validate_summary → branch → success/failure report
```

Database credentials must come from Airflow Variables.

In [9]:
# TODO 1.1
# Create dags/<DAG_ID>.py.
# Include imports, DAG metadata, make_engine(), and a read_config task.
# Use Airflow Variables for DB credentials.

from pathlib import Path
import textwrap

dag_path = PROJECT / "dags" / f"{DAG_ID}.py"

dag_code = r'''
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from urllib.parse import quote_plus

from sqlalchemy import create_engine, text


try:
    from airflow.decorators import dag, task
    from airflow.models import Variable
except ImportError:
    from airflow.sdk import dag, task, Variable


DAG_ID = "__DAG_ID__"
STUDENT_SCHEMA = "__STUDENT_SCHEMA__"
SUMMARY_VIEW = f"{STUDENT_SCHEMA}.airbnb_neighbourhood_summary"


def make_engine():
    """
    Create a PostgreSQL connection from Airflow Variables.

    Important:
    No username, password, host, database URL, or private credential is hard-coded.
    """

    host = Variable.get("QBC12_DB_HOST")
    port = Variable.get("QBC12_DB_PORT", default_var="5432")
    database = Variable.get("QBC12_DB_NAME")
    username = Variable.get("QBC12_DB_USER")
    password = Variable.get("QBC12_DB_PASSWORD")

    url = (
        "postgresql+psycopg2://"
        f"{quote_plus(username)}:{quote_plus(password)}"
        f"@{host}:{port}/{database}"
    )

    return create_engine(url, pool_pre_ping=True)


@dag(
    dag_id=DAG_ID,
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["qbc12", "hw01", "airbnb"],
)
def airbnb_pipeline():

    @task
    def read_config() -> dict:
        """
        Read non-secret pipeline configuration.

        The source table names are configurable through Airflow Variables.
        The default names are placeholders and can be overridden in Airflow.
        """

        return {
            "student_schema": STUDENT_SCHEMA,
            "summary_view": SUMMARY_VIEW,
            "source_listings_table": Variable.get(
                "QBC12_AIRBNB_LISTINGS_TABLE",
                default_var="public.listings_sample",
            ),
            "source_segments_table": Variable.get(
                "QBC12_AIRBNB_SEGMENTS_TABLE",
                default_var="public.neighbourhood_segments",
            ),
        }

    @task
    def refresh_summary(config: dict) -> dict:
        engine = make_engine()

        student_schema = config["student_schema"]
        summary_view = config["summary_view"]
        listings_table = config["source_listings_table"]
        segments_table = config["source_segments_table"]

        refresh_sql = f"""
        CREATE SCHEMA IF NOT EXISTS {student_schema};

        DROP MATERIALIZED VIEW IF EXISTS {summary_view};

        CREATE MATERIALIZED VIEW {summary_view} AS
        WITH neighbourhood_summary AS (
            SELECT
                neighbourhood,
                COUNT(listing_id) AS num_listings,
                AVG(price)::numeric(12, 2) AS avg_price,
                PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY price)::numeric(12, 2) AS median_price,
                AVG(minimum_nights)::numeric(12, 2) AS avg_minimum_nights,
                AVG(availability_365)::numeric(12, 2) AS availability_365_avg,
                SUM(number_of_reviews) AS total_reviews,
                (
                    SUM(number_of_reviews)::numeric
                    / NULLIF(COUNT(listing_id), 0)
                )::numeric(12, 2) AS reviews_per_listing
            FROM {listings_table}
            GROUP BY neighbourhood
        )
        SELECT
            ns.neighbourhood,
            ns.num_listings,
            ns.avg_price,
            ns.median_price,
            ns.avg_minimum_nights,
            ns.availability_365_avg,
            ns.total_reviews,
            ns.reviews_per_listing,
            COALESCE(seg.tourism_segment, 'unknown') AS tourism_segment,
            COALESCE(seg.priority_level, 'unknown') AS priority_level
        FROM neighbourhood_summary ns
        LEFT JOIN {segments_table} seg
            ON ns.neighbourhood = seg.neighbourhood
        ORDER BY ns.neighbourhood;

        CREATE INDEX IF NOT EXISTS idx_airbnb_neighbourhood_summary_neighbourhood
        ON {summary_view} (neighbourhood);

        CREATE INDEX IF NOT EXISTS idx_airbnb_neighbourhood_summary_priority
        ON {summary_view} (priority_level);
        """

        with engine.begin() as conn:
            conn.execute(text(refresh_sql))

        return {
            "status": "refreshed",
            "refreshed_object": summary_view,
        }

    @task
    def validate_summary(config: dict, refresh_result: dict) -> dict:
        """
        Validate the materialized view.

        Required HW01-C checks:
        - row_count > 0
        - null_neighbourhoods == 0
        - bad_prices == 0
        - bad_availability == 0
        """

        engine = make_engine()
        summary_view = config["summary_view"]

        validation_sql = f"""
        SELECT
            COUNT(*) AS row_count,

            COUNT(*) FILTER (
                WHERE neighbourhood IS NULL
            ) AS null_neighbourhoods,

            COUNT(*) FILTER (
                WHERE avg_price IS NULL
                   OR avg_price < 0
                   OR median_price IS NULL
                   OR median_price < 0
            ) AS bad_prices,

            COUNT(*) FILTER (
                WHERE availability_365_avg IS NULL
                   OR availability_365_avg < 0
                   OR availability_365_avg > 365
            ) AS bad_availability
        FROM {summary_view};
        """

        with engine.begin() as conn:
            row = conn.execute(text(validation_sql)).mappings().one()

        result = dict(row)

        result["passed"] = (
            result["row_count"] > 0
            and result["null_neighbourhoods"] == 0
            and result["bad_prices"] == 0
            and result["bad_availability"] == 0
        )

        result["refreshed_object"] = refresh_result["refreshed_object"]

        return result

    @task.branch
    def choose_report_path(validation_result: dict) -> str:
        if validation_result["passed"]:
            return "write_success_report"

        return "write_failure_report"

    @task
    def write_success_report(config: dict, validation_result: dict) -> str:
        report_path = Path("/tmp/hw01_c_airflow_success.md")

        report = f"""# HW01-C Airflow Run Report

Status: SUCCESS

DAG ID: {DAG_ID}

Refreshed object: {validation_result["refreshed_object"]}

Validation result:

- row_count: {validation_result["row_count"]}
- null_neighbourhoods: {validation_result["null_neighbourhoods"]}
- bad_prices: {validation_result["bad_prices"]}
- bad_availability: {validation_result["bad_availability"]}
- passed: {validation_result["passed"]}

Credentials are loaded from Airflow Variables.
No credentials are hard-coded in the DAG.
"""

        report_path.write_text(report, encoding="utf-8")
        return str(report_path)

    @task
    def write_failure_report(config: dict, validation_result: dict) -> str:
        report_path = Path("/tmp/hw01_c_airflow_failure.md")

        report = f"""# HW01-C Airflow Run Report

Status: FAILURE

DAG ID: {DAG_ID}

Refreshed object: {validation_result["refreshed_object"]}

Validation result:

- row_count: {validation_result["row_count"]}
- null_neighbourhoods: {validation_result["null_neighbourhoods"]}
- bad_prices: {validation_result["bad_prices"]}
- bad_availability: {validation_result["bad_availability"]}
- passed: {validation_result["passed"]}

The DAG failed because one or more validation checks did not pass.
"""

        report_path.write_text(report, encoding="utf-8")

        raise ValueError(f"Validation failed: {validation_result}")

    config = read_config()
    refresh_result = refresh_summary(config)
    validation_result = validate_summary(config, refresh_result)

    branch = choose_report_path(validation_result)

    success_report = write_success_report(config, validation_result)
    failure_report = write_failure_report(config, validation_result)

    branch >> [success_report, failure_report]


airbnb_pipeline()
'''

dag_code = dag_code.replace("__DAG_ID__", DAG_ID)
dag_code = dag_code.replace("__STUDENT_SCHEMA__", STUDENT_SCHEMA)

dag_path.write_text(textwrap.dedent(dag_code).strip() + "\n", encoding="utf-8")

print(f"Created DAG file: {dag_path}")

Created DAG file: /home/alireza/PycharmProjects/Quera-Homeworks/HW01-C/dags/qbc12_hw01_alirezaabavi_airbnb_pipeline.py


## 2. Refresh task

The refresh task should recreate your materialized view in Postgres. Do not move the full dataset into Python.

In [10]:
# TODO 2.1
# Add refresh_summary(config).
# It should create/refresh your materialized view and indexes.
# Return a small dict, not a dataframe.

dag_text = dag_path.read_text(encoding="utf-8")

assert "def refresh_summary" in dag_text, "refresh_summary task is missing"
assert "CREATE MATERIALIZED VIEW" in dag_text, "Materialized view SQL is missing"
assert "neighbourhood_summary" in dag_text, "summary logic is missing"
assert "AVG(price)" in dag_text, "avg_price calculation is missing"
assert "PERCENTILE_CONT" in dag_text, "median_price calculation is missing"
assert "SUM(number_of_reviews)" in dag_text, "total_reviews calculation is missing"
assert "LEFT JOIN" in dag_text, "segments join is missing"
assert "CREATE INDEX" in dag_text, "index creation is missing"

print("TODO 2.1 complete: refresh_summary exists and mirrors summary logic.")

TODO 2.1 complete: refresh_summary exists and mirrors HW01-A summary logic.


## 3. Validation task

Required checks:

- row_count > 0
- null_neighbourhoods == 0
- bad_prices == 0
- bad_availability == 0

In [11]:
# TODO 4.1
# Add choose_report_path, write_success_report, and write_failure_report.
# Failure report should raise ValueError after writing the report.

dag_text = dag_path.read_text(encoding="utf-8")

assert "@task.branch" in dag_text, "Airflow branch task is missing"
assert "def choose_report_path" in dag_text, "choose_report_path is missing"
assert "def write_success_report" in dag_text, "write_success_report is missing"
assert "def write_failure_report" in dag_text, "write_failure_report is missing"
assert "raise ValueError" in dag_text, "failure report should raise ValueError"

print("TODO 4.1 complete: branching and success/failure reports exist.")

TODO 4.1 complete: branching and success/failure reports exist.


## 4. Branching and reports

Success and failure should not look the same.

Use `@task.branch` to choose the report path.

In [12]:
# TODO 4.1
# Add choose_report_path, write_success_report, and write_failure_report.
# Failure report should raise ValueError after writing the report.

dag_text = dag_path.read_text(encoding="utf-8")

assert "@task.branch" in dag_text, "Airflow branch task is missing"
assert "def choose_report_path" in dag_text, "choose_report_path is missing"
assert "def write_success_report" in dag_text, "write_success_report is missing"
assert "def write_failure_report" in dag_text, "write_failure_report is missing"
assert "raise ValueError" in dag_text, "failure report should raise ValueError"

print("TODO 4.1 complete: branching and success/failure reports exist.")

TODO 4.1 complete: branching and success/failure reports exist.


In [13]:
# Syntax check. This is not a full Airflow run.
import py_compile

dag_path = PROJECT / 'dags' / f'{DAG_ID}.py'
assert dag_path.exists(), f'Missing DAG file: {dag_path}'
py_compile.compile(str(dag_path), doraise=True)
print('DAG compiles:', dag_path)

DAG compiles: /home/alireza/PycharmProjects/Quera-Homeworks/HW01-C/dags/qbc12_hw01_alirezaabavi_airbnb_pipeline.py


## 5. Shared Airflow run

In shared Airflow:

1. find your DAG
2. unpause it
3. trigger it manually
4. inspect Graph view
5. inspect logs
6. confirm the materialized view was refreshed

Screenshots:

```text
screenshots/airflow_dag_graph.png
screenshots/airflow_success_run.png
```

In [ ]:
# TODO 5.1
# Write reports/hw01_c_airflow.md.
# Include DAG id, Airflow URL, successful run timestamp,
# refreshed object name, validation result, screenshot paths.

# Write your code here.

In [ ]:
for file in [f'dags/{DAG_ID}.py', 'reports/hw01_c_airflow.md']:
    assert Path(file).exists(), f'Missing {file}'
print('Basic deliverables exist.')

## Commit

```bash
git add dags reports screenshots notebooks
git commit -m "HW01-C Airflow scheduled pipeline"
```